# 01.2 — Regex for NLP

**Problem it solves:** Extracting structured information from raw text without a full parser. Emails, URLs, dates, mentions, numbers, phone numbers — all pattern-matchable with regex.

**Why it matters for NLP:** Regex is the first line of text preprocessing. Tokenizers, sentence segmenters, and named entity extractors all use regex under the hood. If you can't write regex, you're debugging other people's patterns blind.

**What we'll build:** A production-grade text extraction pipeline — the kind that runs before any model sees the text.

---

## Part 1: Regex Foundations — the NLP-relevant subset

In [ ]:
import re

# The patterns you actually use in NLP:
# \w  = word character [a-zA-Z0-9_]
# \W  = non-word character
# \b  = word boundary (zero-width assertion)
# \d  = digit [0-9]
# \s  = whitespace [\t\n\r\f\v ]
# .   = any char except newline (use re.DOTALL to include newline)
# *   = 0 or more (greedy)
# +   = 1 or more (greedy)
# ?   = 0 or 1 (greedy)
# *?  = 0 or more (lazy — stops at first match)
# (?:...) = non-capturing group
# (?P<name>...) = named capture group
# (?=...) = lookahead (match if followed by...)
# (?<=...) = lookbehind (match if preceded by...)

# Word boundary demo — crucial for NLP
text = "cats concatenate catfish"
print("With \\b:   ", re.findall(r'\bcat\b', text))   # only standalone 'cat'
print("Without \\b:", re.findall(r'cat', text))        # all occurrences

In [ ]:
# Greedy vs lazy — a common NLP gotcha
html = '<b>bold</b> and <i>italic</i>'

greedy = re.findall(r'<.+>', html)    # greedy: matches as much as possible
lazy   = re.findall(r'<.+?>', html)   # lazy:   matches as little as possible

print(f"Greedy: {greedy}")
print(f"Lazy:   {lazy}")

In [ ]:
# Named groups — make complex patterns readable
date_pattern = r'(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})'
m = re.search(date_pattern, 'Published on 2024-03-15 at 9am')
if m:
    print(m.groupdict())
    print(f"Year: {m.group('year')}, Month: {m.group('month')}, Day: {m.group('day')}")

## Part 2: Build a Text Entity Extractor

In [ ]:
# Build each pattern, understand the tradeoffs

# --- EMAIL ---
# Naive version
email_naive = r'\S+@\S+'
# Better: RFC-5321 is 6 pages long. This covers ~99% of real emails:
email_pattern = r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}'

test_emails = [
    'Contact us at hello@example.com or support@company.co.uk',
    'Bad emails: @nodomain, noatsign.com, double@@sign.com',
    'Subaddressing: user+tag@gmail.com',
]
for t in test_emails:
    found = re.findall(email_pattern, t)
    print(f"Input:  {t}")
    print(f"Found:  {found}")
    print()

In [ ]:
# --- URL ---
# URLs are notoriously hard. This handles the common cases:
url_pattern = r'https?://(?:www\.)?[-a-zA-Z0-9@:%._+~#=]{1,256}\.[a-zA-Z0-9()]{1,6}\b(?:[-a-zA-Z0-9()@:%_+.~#?&/=]*)'

test_urls = [
    'Visit https://www.example.com/path?q=hello&page=2 for more',
    'Also see http://subdomain.site.co.uk/article#section',
    'Not a URL: example.com (no scheme), ftp://ignored.com',
]
for t in test_urls:
    found = re.findall(url_pattern, t)
    print(f"Found: {found}")

In [ ]:
# --- TWITTER/SOCIAL MENTIONS & HASHTAGS ---
mention_pattern = r'@([\w]+)'   # captures the name without @
hashtag_pattern = r'#([\w]+)'

tweet = "Hey @johnDoe and @jane_smith, check out #NLP and #MachineLearning!"
print(f"Mentions: {re.findall(mention_pattern, tweet)}")
print(f"Hashtags: {re.findall(hashtag_pattern, tweet)}")

In [ ]:
# --- NUMBERS (with units) ---
# Covers: 42, 3.14, -7, 1,000, $9.99, 15%, 2.5kg
number_pattern = r'[$€£]?-?\d{1,3}(?:,\d{3})*(?:\.\d+)?(?:\s?(?:kg|km|ml|%|USD|EUR))?'

test = 'Price: $1,299.99. Weight: 2.5kg. Discount: 15%. Temperature: -7.3 degrees.'
print(re.findall(number_pattern, test))

## Part 3: Regex for Tokenization

This is how real tokenizers like NLTK's `TweetTokenizer` work under the hood.

In [ ]:
# Naive tokenizer: split on whitespace
text = "I can't wait for the U.S.A. trip — it'll be great!"
print("Whitespace split:", text.split())
# Problems: punctuation glued to words, contractions split wrong, 
# abbreviations broken, em-dash not handled

In [ ]:
# Regex tokenizer: define what a TOKEN is, then find all
# Order matters — patterns are tried left to right, first match wins

TOKEN_PATTERNS = [
    ('URL',         r'https?://\S+'),
    ('EMAIL',       r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}'),
    ('MENTION',     r'@[\w]+'),
    ('HASHTAG',     r'#[\w]+'),
    ('ABBREV',      r'(?:[A-Z]\.){2,}'),               # U.S.A., e.g.
    ('CONTRACTION', r"[\w]+n't|[\w]+'[\w]+"),           # can't, I'll, we're
    ('NUMBER',      r'\d+(?:[.,]\d+)*'),
    ('WORD',        r"[\w']+"),
    ('ELLIPSIS',    r'\.{2,}'),
    ('PUNCT',       r'[^\w\s]'),
]

# Compile into one big alternation pattern
combined = '|'.join(f'(?P<{name}>{pat})' for name, pat in TOKEN_PATTERNS)
compiled = re.compile(combined)

def tokenize(text: str) -> list[dict]:
    tokens = []
    for m in compiled.finditer(text):
        tokens.append({
            'token': m.group(),
            'type': m.lastgroup,
            'span': m.span(),
        })
    return tokens

# Test
test_sentences = [
    "I can't wait for the U.S.A. trip — it'll be great!",
    "Email john@doe.com or visit https://example.com #NLP @user123",
    "Price: $1,299.99 and 15% off... what a deal!",
]

for sent in test_sentences:
    print(f"\nInput: {sent}")
    for tok in tokenize(sent):
        print(f"  {tok['token']:25} {tok['type']}")

## Part 4: Regex for Text Cleaning

In [ ]:
def clean_text(text: str, 
               remove_urls: bool = True,
               remove_html: bool = True,
               remove_mentions: bool = False,
               remove_hashtags: bool = False,
               normalize_whitespace: bool = True) -> str:
    """
    Clean raw text before feeding to a model.
    The boolean flags let callers control the pipeline per use-case.
    """
    if remove_html:
        text = re.sub(r'<[^>]+>', ' ', text)          # strip HTML tags
        text = re.sub(r'&[a-z]+;', ' ', text)         # strip HTML entities (&amp; etc)
    
    if remove_urls:
        text = re.sub(r'https?://\S+|www\.\S+', '[URL]', text)
    
    if remove_mentions:
        text = re.sub(r'@[\w]+', '[USER]', text)
    
    if remove_hashtags:
        text = re.sub(r'#[\w]+', '', text)
    
    if normalize_whitespace:
        text = re.sub(r'\s+', ' ', text).strip()
    
    return text

samples = [
    '<p>Check out <a href="https://example.com">this link</a> &amp; follow @user!</p>',
    'Visit www.example.com for #MachineLearning resources @prof_smith',
    'Text with\n\n\nmultiple   spaces\t\tand tabs',
]

for s in samples:
    print(f"Raw:     {s!r}")
    print(f"Cleaned: {clean_text(s)!r}")
    print()

## Part 5: Sentence Boundary Detection with Regex

The naive approach and why it fails.

In [ ]:
# Naive: split on '.', '!', '?'
def naive_sentence_split(text: str) -> list[str]:
    return re.split(r'[.!?]', text)

hard_text = """Dr. Smith works at Apple Inc. in the U.S.A. 
He earned $2.5M last year. Really? Yes! "No way," she said. 
Visit https://example.com. The end."""

print("=== NAIVE ===")
for i, s in enumerate(naive_sentence_split(hard_text)):
    if s.strip():
        print(f"  [{i}] {s.strip()!r}")

In [ ]:
# Better: use lookahead to split AFTER sentence-ending punctuation
# followed by whitespace and a capital letter (heuristic)

ABBREVS = {'Dr', 'Mr', 'Mrs', 'Ms', 'Prof', 'Sr', 'Jr', 
           'Inc', 'Ltd', 'Corp', 'Co', 'vs', 'etc', 'i.e', 'e.g'}

def regex_sentence_split(text: str) -> list[str]:
    # Normalize newlines
    text = re.sub(r'\n+', ' ', text)
    
    # Split on: . ! ?  followed by space + uppercase
    # but NOT if preceded by a known abbreviation
    candidates = re.split(r'(?<=[.!?])\s+(?=[A-Z"\'])', text)
    
    # Merge back splits that were inside abbreviations
    merged = []
    buffer = ''
    for chunk in candidates:
        if buffer:
            chunk = buffer + ' ' + chunk
            buffer = ''
        # Check if the last 'word' before the split is an abbreviation
        last_word = re.search(r'(\w+)\.?$', chunk.rstrip())
        if last_word and last_word.group(1) in ABBREVS:
            buffer = chunk  # hold — this wasn't really a sentence end
        else:
            merged.append(chunk)
    if buffer:
        merged.append(buffer)
    
    return [s.strip() for s in merged if s.strip()]

print("=== REGEX HEURISTIC ===")
for i, s in enumerate(regex_sentence_split(hard_text)):
    print(f"  [{i}] {s!r}")

## Part 6: What Can't Regex Do?

The fundamental limits.

In [ ]:
# 1. Regex cannot match nested/balanced structures
# Example: matching balanced parentheses
# (a (b (c) d) e) — regex can't verify these balance
# You need a stack/parser for this

# Regex matches the FIRST close paren, not the matching one:
text = '(outer (inner content) still outer)'
print(re.findall(r'\([^)]+\)', text))  # wrong — misses nesting

# 2. Regex cannot understand context/meaning
# "I saw her duck" — is 'duck' a noun or verb? Regex can't tell.
# "The bank by the river" vs "The bank I use" — same word, different sense.

# 3. Regex on Unicode needs re.UNICODE flag (default in Python 3)
# but \w still misses some scripts without re.UNICODE:
arabic = 'مرحبا بالعالم'
print(f"\\w+ on Arabic: {re.findall(r'\\w+', arabic)}")  # works in Py3
print(f"Length: {len(re.findall(r'\\w+', arabic))} words")

## Summary

**What it does:** Pattern-based extraction and transformation of text — fast, interpretable, no model needed.

**When to use it:** Preprocessing (before any model), extracting structured fields (emails, URLs, dates), quick text cleaning, rule-based tokenization.

**What breaks it:**
- Nested structures (HTML, JSON, code) — use a real parser instead
- Ambiguous context — regex sees form, not meaning
- Multilingual text with non-Latin scripts needs care (test your `\w` assumptions)
- Catastrophic backtracking: `(a+)+b` on `aaaaaaac` can freeze a process — always test on adversarial inputs

**Rule of thumb:** If you need to explain the regex in more than 2 lines, you've outgrown regex — use a proper tokenizer or parser.

---
**Next:** 01.3 — Tokenization: rule-based and statistical, building your own tokenizer